# 考える過程を書かせて LLM に算数の文章題を解かせる（Chain-of-Thought と self-consistency）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/21_llm/llm_reasoning.ipynb)

NLP クックブック 21-11 ／ 大規模言語モデル (LLM)

> GPU ランタイム推奨（ランタイム → ランタイムのタイプを変更 → T4 GPU）

算数の文章題のように何段階かの計算が要る問題で、LLM にいきなり答えさせる場合と、途中の考えを書かせてから答えさせる場合 (Chain-of-Thought) とで、正解率がどう変わるかを確かめたい。何回か解かせて多数決をとる方法 (self-consistency) の効果と、生成するトークン数や時間のコストも測りたい。

In [1]:
!pip install -q datasets

In [2]:
import itertools, re, time, unicodedata
from collections import Counter
import numpy as np
import pandas as pd
import sentencepiece as spm
import torch
from datasets import load_dataset
from huggingface_hub import hf_hub_download
from tokenizers import models
from transformers import AutoModelForCausalLM, AutoTokenizer

torch.manual_seed(0)
# MGSM (Shi et al., 2022): GSM8K (Cobbe et al., 2021) の小学校レベルの算数の文章題 250 問を、人手で 10 言語に訳したもの。
# 元のリポジトリ (google-research/url-nlp の mgsm/) のライセンスは CC BY 4.0、元の GSM8K は MIT ライセンス
# (Hugging Face のデータセットカードのメタデータは CC BY-SA 4.0。注意書きを参照)
N = 100                                                    # 時間の都合で 250 問のうち先頭の 100 問を使う
mgsm = load_dataset("juletxara/mgsm", "ja", split="test").select(range(N))
print(mgsm[0]["question"], "→", mgsm[0]["answer_number"])

name = "sbintuitions/sarashina2.2-1b-instruct-v0.1"       # MIT ライセンス
tokenizer = AutoTokenizer.from_pretrained(name, padding_side="left")
# transformers 5 系は、このモデルの SentencePiece (Unigram) を BPE に変換してしまい、学習時と違う分割になる。
# 元の tokenizer.model から語彙とスコアを読み、分割の方式だけを Unigram に戻す (特殊トークンやチャットテンプレートはそのまま)
sp = spm.SentencePieceProcessor(model_file=hf_hub_download(name, "tokenizer.model"))
tokenizer.backend_tokenizer.model = models.Unigram(
    [(sp.id_to_piece(i), sp.get_score(i)) for i in range(sp.get_piece_size())], unk_id=sp.unk_id(), byte_fallback=True)
BF16 = torch.cuda.get_device_capability()[0] >= 8      # bfloat16 は Ampere (A100, L4) 以降。T4 では float16 を使う
model = AutoModelForCausalLM.from_pretrained(name, dtype=torch.bfloat16 if BF16 else torch.float16, device_map="auto").eval()
print(model.generation_config)      # 開発元の既定値: 温度 0.7、top_p 0.9、repetition_penalty 1.05

ジャネットのアヒルは1日に16個の卵を生みます。ジャネットは毎朝朝食の一環で3個を消費し、毎日4個使って友達向けにマフィンを焼きます。残りを市場で1個あたり2ドルの価格で売ります。彼女は毎日市場でいくら手に入れていますか？ → 18
GenerationConfig {
  "bos_token_id": 1,
  "do_sample": true,
  "eos_token_id": 2,
  "repetition_penalty": 1.05,
  "temperature": 0.7,
  "top_p": 0.9
}



In [3]:
# few-shot の例: MGSM に付いている、人手で訳した 8 問の例 (train) の先頭 2 問。答えの部分だけを変えて「直接」と「CoT」の 2 通りにする
# (2 問目の問題文の「難題のコンピューター」は「何台の」の誤りで、元のデータのまま使っている)
exemplars = load_dataset("juletxara/mgsm", "ja", split="train").select(range(2))
INSTRUCTION = {
    "直接": "次の問題に答えてください。答えの数値だけを出力し、途中の考えや説明は書かないでください。",
    "CoT": "次の問題に答えてください。ステップごとに考えて途中の計算を書き、最後に「答えは〇〇です。」と書いてください。",
}
def shot_answer(style, ex):
    if style == "直接":
        return f"答えは{ex['answer_number']}です。"
    return ex["answer"].removeprefix("ステップごとの答え：")      # 例に付いている推論の過程 (「…。答えは11です。」で終わる)

def messages(style, n_shots, question):
    msgs = []
    for ex in exemplars.select(range(n_shots)):     # 例は「ユーザーの問題 → アシスタントの答え」の対話として前に置く
        msgs += [{"role": "user", "content": f"{INSTRUCTION[style]}\n\n問題: {ex['question'].removeprefix('問題：')}"},
                 {"role": "assistant", "content": shot_answer(style, ex)}]
    return msgs + [{"role": "user", "content": f"{INSTRUCTION[style]}\n\n問題: {question}"}]

@torch.no_grad()
def solve(style, n_shots, max_new_tokens, n_samples=1, bs=64):
    """全問を解かせ、問題ごとに n_samples 個の出力と、生成したトークン数・時間を返す。n_samples > 1 ならサンプリング、1 なら貪欲法"""
    texts = [tokenizer.apply_chat_template(messages(style, n_shots, ex["question"]), add_generation_prompt=True, tokenize=False) for ex in mgsm]
    outs, n_tokens, t = [], [], time.time()
    step = max(1, bs // n_samples)
    for i in range(0, len(texts), step):
        enc = tokenizer(texts[i:i + step], return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=n_samples > 1, num_return_sequences=n_samples,
                             pad_token_id=tokenizer.pad_token_id)[:, enc["input_ids"].shape[1]:]
        n_tokens += (out != tokenizer.pad_token_id).sum(1).tolist()     # 終了の記号 </s> までを数える
        outs += tokenizer.batch_decode(out, skip_special_tokens=True)
    return {"text": np.array(outs, dtype=object).reshape(N, n_samples), "tokens": np.array(n_tokens).reshape(N, n_samples),
            "sec": time.time() - t, "max_new_tokens": max_new_tokens}

def extract(text):
    """出力から答えの数値を取り出す: 「答え」の後ろ → \\boxed{} の中 → 最後に出てきた数値 の順に探す"""
    text = unicodedata.normalize("NFKC", text).replace(",", "")
    text = re.sub(r"(\d+(?:\.\d+)?)万(\d*)", lambda m: str(round(float(m.group(1)) * 10000) + int(m.group(2) or 0)), text)   # 「3万5000」→ 35000
    for pattern in [r"答え[はが]?\s*[:：]?[^\d\-\n]*(-?\d+(?:\.\d+)?)", r"\\boxed\{[^\d\-]*(-?\d+(?:\.\d+)?)"]:
        found = re.findall(pattern, text)
        if found:
            return float(found[-1])
    found = re.findall(r"-?\d+(?:\.\d+)?", text)
    return float(found[-1]) if found else None

gold = np.array(mgsm["answer_number"], dtype=float)
print([extract(s) for s in ["答えは65,000ドルです。", "よって \\boxed{18} です。", "合計は3万ドルです。", "3万5000円", "わかりません"]])

[65000.0, 18.0, 30000.0, 35000.0, None]


In [4]:
# 貪欲法で 1 回ずつ解かせる。「直接」は 16 トークン、CoT は 400 トークンまで (Colab の T4 で 3〜4 分程度)
CONDITIONS = {"直接 (0-shot)": ("直接", 0, 16), "直接 (2-shot)": ("直接", 2, 16), "CoT (0-shot)": ("CoT", 0, 400), "CoT (2-shot)": ("CoT", 2, 400)}
runs = {c: solve(*args) for c, args in CONDITIONS.items()}

def summary(run, answers):
    return {"正解率": (answers == gold).mean(), "答えを取り出せなかった割合": np.isnan(answers).mean(),
            "出力のトークン数 (平均)": run["tokens"].mean(), "上限で打ち切られた割合": (run["tokens"] >= run["max_new_tokens"]).mean(),
            "1 問あたりの秒数": run["sec"] / N}

greedy = {c: np.array([extract(t) for t in r["text"][:, 0]], dtype=float) for c, r in runs.items()}   # 取り出せなければ nan (不正解)
pd.set_option("display.width", 200)
print(pd.DataFrame({c: summary(runs[c], greedy[c]) for c in runs}).round(3).to_string())
print(f"GPU メモリ最大 {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")
for c in runs:
    print(f"--- {c} (正解 {gold[1]:g}):", runs[c]["text"][1, 0][-300:])
print("--- 直接 (2-shot) で数値を取り出せなかった出力:", [t for t, a in zip(runs["直接 (2-shot)"]["text"][:, 0], greedy["直接 (2-shot)"]) if np.isnan(a)][:3])

               直接 (0-shot)  直接 (2-shot)  CoT (0-shot)  CoT (2-shot)
正解率                  0.020        0.070         0.350          0.41
答えを取り出せなかった割合        0.310        0.060         0.000          0.00
出力のトークン数 (平均)       16.000       16.000       170.980        154.64
上限で打ち切られた割合          1.000        1.000         0.020          0.01
1 問あたりの秒数            0.026        0.018         0.215          0.32
GPU メモリ最大 7.3 GB
--- 直接 (0-shot) (正解 3): 問題文より、青色の繊維を2巻、白色の繊維はその半分
--- 直接 (2-shot) (正解 3): 答えは4巻です。青色の繊維を2巻必要とし、白色の
--- CoT (0-shot) (正解 3): 与えられた問題を解くために、まず青色の繊維と白色の繊維それぞれに必要な巻数を計算します。

青色の繊維は2巻分必要なので、これはそのまま2巻です。
次に白色の繊維ですが、こちらは青色の繊維の半分を用いる必要があるとのことなので、青色の繊維2巻の半分である1巻が必要となります。

したがって、ローブを作成するために必要な繊維の総巻数は、青色の繊維2巻と白色の繊維1巻を合わせた3巻となります。

よって、全体で必要な巻数は「3巻」です。

「答えは3巻です。」
--- CoT (2-shot) (正解 3): まず、青色の繊維を2巻必要です。次に、白色の繊維は青色の繊維の半分なので、2/2=1巻分です。合計で必要な巻数は、青色の繊維と白色の繊維を足した数になります。つまり、2+1=3巻です。答えは3です。
--- 直接 (2-shot) で数値を取り出せなかった出力: ['総額を計算してみましょう。まず、通常の価格で購入するグラスの費用を計算します。', 'この問題では、仕入れ計画ごとの利益を

In [5]:
# self-consistency: 温度 0.7 でサンプリングして 5 回ずつ解かせ、取り出した答えの多数決をとる (Colab の T4 で 6〜7 分程度)
K = 5
torch.manual_seed(0)
samples = {c: solve(*CONDITIONS[c], n_samples=K) for c in ["直接 (2-shot)", "CoT (0-shot)"]}
answers = {c: [[extract(t) for t in row] for row in r["text"]] for c, r in samples.items()}

def vote(row):
    """答えを取り出せたものだけで多数決。同数なら先に出た答え"""
    row = [a for a in row if a is not None]
    return Counter(row).most_common(1)[0][0] if row else None

def sc_accuracy(c, k):
    """K 個から k 個を選ぶすべての組み合わせで多数決をとり、正解率を平均する (どの k 個を使うかによるばらつきをならす)"""
    combos = list(itertools.combinations(range(K), k))
    return np.mean([[vote([row[j] for j in combo]) == g for combo in combos] for row, g in zip(answers[c], gold)])

table = pd.DataFrame({f"{c} の多数決": {k: sc_accuracy(c, k) for k in range(1, K + 1)} for c in samples})
table.index.name = "回数"
print(table.round(3).to_string())
print({c: f"5 回で 1 問あたり {r['tokens'].sum(1).mean():.0f} トークン、{r['sec'] / N:.2f} 秒" for c, r in samples.items()})

    直接 (2-shot) の多数決  CoT (0-shot) の多数決
回数                                     
1              0.066              0.290
2              0.062              0.297
3              0.065              0.330
4              0.062              0.348
5              0.070              0.360
{'直接 (2-shot)': '5 回で 1 問あたり 80 トークン、0.09 秒', 'CoT (0-shot)': '5 回で 1 問あたり 930 トークン、1.27 秒'}


In [6]:
# 差は偶然の範囲か: 同じ問題で比べる対応のあるブートストラップで、正解率の差の 95% 信頼区間を求める
rng = np.random.default_rng(0)
correct = {c: greedy[c] == gold for c in runs}
correct["CoT (0-shot) 5 回の多数決"] = np.array([vote(row) == g for row, g in zip(answers["CoT (0-shot)"], gold)])
# サンプリング 1 回の正解は、5 個の標本の正解の平均 (問題ごとに 0〜1 の値) にする。多数決の表の 1 回の行と同じ基準
correct["CoT (0-shot) サンプリング 1 回"] = np.array([np.mean([a == g for a in row]) for row, g in zip(answers["CoT (0-shot)"], gold)])

def diff_ci(a, b, n_boot=2000):
    x, y = correct[a], correct[b]
    idx = rng.integers(0, N, (n_boot, N))
    d = x[idx].mean(axis=1) - y[idx].mean(axis=1)
    print(f"{a} − {b}: {(x.mean() - y.mean()) * 100:+5.1f} ポイント [{np.percentile(d, 2.5) * 100:+5.1f}, {np.percentile(d, 97.5) * 100:+5.1f}]")

diff_ci("CoT (2-shot)", "直接 (2-shot)")          # 同じ 2 問の例で、答えの前に推論を見せるかどうかだけが違う
diff_ci("CoT (0-shot)", "直接 (2-shot)")
diff_ci("CoT (0-shot)", "CoT (2-shot)")
diff_ci("CoT (0-shot) 5 回の多数決", "CoT (0-shot)")
diff_ci("CoT (0-shot) 5 回の多数決", "CoT (0-shot) サンプリング 1 回")

# 票の集まり方は確信度の目安になる: 5 回のうち最も多かった答えが何票だったかごとの、多数決の正解率
top = np.array([max(Counter(a for a in row if a is not None).values(), default=0) for row in answers["CoT (0-shot)"]])
print(pd.DataFrame({"問題数": pd.Series(top).value_counts().sort_index(),
                    "多数決の正解率": pd.Series(correct["CoT (0-shot) 5 回の多数決"]).groupby(top).mean().round(3)}).rename_axis("最多票の数").to_string())

CoT (2-shot) − 直接 (2-shot): +34.0 ポイント [+24.0, +45.0]
CoT (0-shot) − 直接 (2-shot): +28.0 ポイント [+18.0, +38.0]
CoT (0-shot) − CoT (2-shot):  -6.0 ポイント [-17.0,  +4.0]
CoT (0-shot) 5 回の多数決 − CoT (0-shot):  +1.0 ポイント [ -9.0, +11.0]
CoT (0-shot) 5 回の多数決 − CoT (0-shot) サンプリング 1 回:  +7.0 ポイント [ +2.2, +12.0]
       問題数  多数決の正解率
最多票の数              
1       31    0.000
2       35    0.314
3       17    0.529
4       12    0.917
5        5    1.000


- Chain-of-Thought (CoT) は、答えの前に途中の推論を書かせるプロンプトの方法で、Wei et al. (2022) は推論の過程を含む例を few-shot で見せる形で提案した。「ステップごとに考えて」のような指示だけで書かせる方法は zero-shot CoT (Kojima et al., 2022) と呼ばれる。ここでは同じ 2 問の例で答えだけを見せると正解率は 7.0%、推論の過程も見せると 41.0% で、差は +34.0 ポイント [+24.0, +45.0] だった。例なしの CoT (35.0%) も、例ありの直接 (7.0%) より 28.0 ポイント [18.0, 38.0] 高かった。例ありの CoT と例なしの CoT の差（6.0 ポイント [−4.0, +17.0]）は、この件数では偶然の範囲である。
- 指示チューニング済みのモデルは「数値だけを出力」と指示しても説明から書き始め、例なしの直接では 16 トークンの中に数値が出てこない出力が 31% あった。例を 2 問見せると多くは「答えは…です。」で書き始めるようになるが、そのあとに説明を続ける（出力の全部が 16 トークンで打ち切られ、6% は数値が出てこなかった）。答えの取り出し方は、取り出せなかった出力を目で確かめて直す。ここでは「答え」の後ろ、\boxed{} の中、最後に出た数値の順に探し、「3万」「3万5000」のような書き方も数値に直している。
- 多数決 (self-consistency) は Wang et al. (2023) の方法で、サンプリングで複数の推論を生成させ、最終的な答えだけで多数決をとる。例なしの CoT で、サンプリング 1 回では 29.0%（5 回のどれを使うかで平均した値）、5 回の多数決で 36.0% になり、差は +7.0 ポイント [+2.2, +12.0] だった。ただし貪欲法 1 回 (35.0%) との差は +1.0 ポイント [−9.0, +11.0] で、このモデルでは 5 倍のコストをかけても貪欲法を上回ったとはいえない。サンプリングは 1 回ごとには貪欲法より正解率が下がり、多数決はその分を取り戻している。推論を書かせない直接の答えで多数決をとっても 6.6% → 7.0% とほとんど変わらなかった。
- 票の集まり方は確信度の目安になる。5 回とも同じ答えだった 5 問では多数決の正解率が 100%、4 票だった 12 問では 91.7% だったのに対し、最多票が 1（同じ答えが 2 回以上出なかった）の 31 問では 0% だった。票が割れた問題だけ人が確かめる、答えを保留する、といった使い方ができる。
- コスト: 出力のトークン数は、直接の 16 トークン（上限）に対して CoT は平均 171.0 トークン、5 回の多数決では 1 問あたり 930 トークンになる。検証に使った GPU（T4 より速いもの。1 回のバッチは 64 件、多数決では 12 問 × 5 回の 60 件）での 1 問あたりの時間は、直接 0.03 秒以下、CoT 0.22 秒、5 回の多数決 1.27 秒だった（T4 ではもっとかかる）。例ありの CoT は出力が短い（平均 154.6 トークン）のに 0.32 秒かかった。例の分だけ入力が長くなるためと考えられる（時間の内訳は測っていない）。バッチの中で最も長い出力が終わるまで待つので、時間は出力の長さのばらつきにも左右される。CoT の 1〜2% は 400 トークンの上限で打ち切られた。
- MGSM は 250 問しかなく、ここではさらに先頭の 100 問を使ったので、正解率の差の信頼区間の幅は 10〜21 ポイントあった。GSM8K と MGSM は公開から時間がたっており、sarashina2.2 の事前学習モデルのモデルカードによると、数学の性能を上げるための合成データでも学習している。評価データやその言い換えが学習データに含まれていれば、点数は実力より高く出る（LLM の評価と分析の章の「日本語のベンチマークで複数の LLM を同じ条件で比べる」の注意書きを参照）。データセットのライセンスは、元のリポジトリの mgsm/LICENSE では CC BY 4.0、Hugging Face のデータセットカードのメタデータでは CC BY-SA 4.0 となっている。
- このモデルの generation_config は、サンプリングの既定値を温度 0.7、top_p 0.9 にし、repetition_penalty 1.05 を貪欲法でも掛ける。設定を変えると点数も変わるので、比べるときはそろえる。トークナイザーの修正については、この章の「オープンな LLM とチャットする（チャットテンプレート）」の注意書きを参照。